# v9 -> Heretic (T4), resumable

Abliterates merged **v9** with [Heretic](https://github.com/p-e-w/heretic).

**Resumes automatically** from `model-training/v9-heretic/study-checkpoint.jsonl`.
Already at ~trial 60/100, so this needs ~40 trials (~20 min) once merged.

**Verified working pieces:** numpy pin, torchao removal, the merge recipe, and the
interactive-menu driving (Heretic 1.4.0 is prompt-only).

Runtime > Change runtime type > **T4 GPU**.


In [ ]:
# 1. Preflight
import subprocess, torch, shutil
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True,text=True).stdout.strip())
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print("torch", torch.__version__, "| cuda", torch.version.cuda)
d=shutil.disk_usage("/content"); print(f"disk: {d.free/1e9:.1f} GB free")
assert d.free > 25e9, "need ~25 GB"


In [ ]:
# 2. Install + fix Colab's dependency soup
# - torchao 0.10 conflicts with peft (raises on import)
# - scikit-learn breaks transformers' optional import on this image
# - numpy 2.1.3 on disk is incompatible with torch 2.11 compiled extensions
!pip -q install uv
!uv pip install --system -q -U heretic-llm "torch==2.11.0" "peft>=0.13" accelerate safetensors pexpect
!python -m pip uninstall -y torchao scikit-learn
import glob, shutil, subprocess, os
SP="/usr/local/lib/python3.13/dist-packages"
for pat in ["numpy","numpy-*.dist-info","numpy.libs"]:
    for p in glob.glob(os.path.join(SP,pat)): shutil.rmtree(p, ignore_errors=True)
!uv pip install --system --no-cache -q "numpy==2.2.6"

# verify in a FRESH interpreter (the kernel caches the old numpy)
r=subprocess.run(["/usr/bin/python3","-c",
  "import numpy,torch,transformers,peft;print(numpy.__version__,torch.__version__,transformers.__version__,peft.__version__)"],
  capture_output=True,text=True)
print(r.stdout.strip() or r.stderr[-500:])
assert r.returncode==0, "environment broken - see above"


In [ ]:
# 3. Mount Drive
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/model-training")
ADAPTER = DRIVE/"v9"/"adapter"
assert ADAPTER.exists(), f"missing {ADAPTER}"
HERE = DRIVE/"v9-heretic"; HERE.mkdir(parents=True, exist_ok=True)
CKPT = HERE/"study-checkpoint.jsonl"
print("adapter:", ADAPTER)
print("checkpoint:", CKPT, CKPT.stat().st_size if CKPT.exists() else "MISSING")


In [ ]:
# 4. Merge v9 (base + LoRA) -> /content/v9-merged
import time, json, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED = Path("/content/v9-merged")
if (MERGED/"model.safetensors").exists() or any(MERGED.glob("*.safetensors")):
    print("already merged")
else:
    t0=time.time()
    m=AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B", dtype=torch.float16,
        device_map={"":0}, low_cpu_mem_usage=True)
    m=PeftModel.from_pretrained(m, str(ADAPTER)); m=m.merge_and_unload()
    m.save_pretrained(MERGED, safe_serialization=True)
    AutoTokenizer.from_pretrained(str(ADAPTER)).save_pretrained(MERGED)
    # Qwen3 ships extra_special_tokens as a LIST; transformers 4.x expects a dict
    tp=MERGED/"tokenizer_config.json"; cfg=json.loads(tp.read_text())
    if isinstance(cfg.get("extra_special_tokens"), list):
        cfg["extra_special_tokens"]={t:t for t in cfg["extra_special_tokens"] if isinstance(t,str)}
        tp.write_text(json.dumps(cfg, indent=2))
    print(f"merged in {time.time()-t0:.0f}s")
    del m; torch.cuda.empty_cache()


In [ ]:
# 5. Restore the optimization checkpoint so trials resume
import shutil
from pathlib import Path
ST = Path("/content/study"); ST.mkdir(exist_ok=True)
if CKPT.exists():
    shutil.copy2(CKPT, ST/"--content--v9-merged.jsonl")
    print("resumed from checkpoint,", CKPT.stat().st_size, "bytes")
else:
    print("no checkpoint - starting fresh (100 trials)")


In [ ]:
# 6. Run Heretic. 1.4.0 is prompt-only, so drive its menus with a PTY.
# Handles: resume menu -> 1, trial menu -> first, action -> Save, export -> Adapter.
import pexpect, time, pathlib

log = open("/content/heretic.log","w")
def w(*a):
    s=" ".join(str(x) for x in a); print(s, flush=True); log.write(s+"\n"); log.flush()

cmd=["heretic","--model","/content/v9-merged",
     "--study-checkpoint-dir","/content/study",
     "--n-trials","100","--n-startup-trials","30"]
w("$"," ".join(cmd))
c=pexpect.spawn(cmd[0], cmd[1:], cwd="/content", encoding="utf-8",
                timeout=10800, codec_errors="replace", dimensions=(50,140))
c.logfile_read=log
try:
    i=c.expect([r"How would you like to proceed\?",
                r"Which trial do you want to use\?",
                r"Determining optimal batch size"], timeout=1800)
    if i==0:
        w("[driver] resume -> 1"); c.sendline("1")
    c.expect(r"Which trial do you want to use\?"); w("[driver] pick first trial"); c.sendline("")
    c.expect(r"What do you want to do with the decensored model\?"); w("[driver] Save"); c.sendline("")
    c.expect(r"Path to the folder:"); w("[driver] path"); c.sendline("/content/v9-heretic")
    j=c.expect([r"How do you want to export the model\?", r"Model saved to",
                pexpect.EOF, pexpect.TIMEOUT], timeout=3600)
    if j==0:
        w("[driver] export -> adapter"); c.send("\x1b[B"); time.sleep(1); c.sendline("")
        c.expect(r"Model saved to", timeout=3600)
    w("[driver] SAVED")
except Exception as e:
    w("[driver] ERROR", type(e).__name__, str(e)[:300])
    raise
finally:
    try: c.close(force=True)
    except Exception: pass


In [ ]:
# 7. Copy the result to Drive
import shutil, time
from pathlib import Path
SRC=Path("/content/v9-heretic"); DST=Path("/content/drive/MyDrive/model-training/v9-heretic/final")
files=[p for p in SRC.rglob("*") if p.is_file()]
print("files:", len(files), [p.name for p in files][:8])
print("size: %.1f MB" % (sum(p.stat().st_size for p in files)/1e6))

t0=time.time()
tmp=DST.with_name("final.part")
shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(SRC, tmp)
shutil.rmtree(DST, ignore_errors=True)
tmp.rename(DST)
print(f"copied to {DST} in {time.time()-t0:.0f}s")

# save the updated optuna journal too, so a future run can continue
shutil.copy2("/content/study/--content--v9-merged.jsonl",
             "/content/drive/MyDrive/model-training/v9-heretic/study-checkpoint.jsonl")
print("journal saved")


## Optional: GGUF for Ollama

If `export -> Adapter` was chosen, the result is a LoRA. To serve it, merge it into
`/content/v9-merged` first, then convert. If you picked Merge instead, convert directly.


In [ ]:
# 8. If the export was an ADAPTER, merge it into a full model first
import subprocess, sys
from pathlib import Path

SRC = Path("/content/v9-heretic")
if (SRC / "adapter_config.json").exists():
    lines = [
        "import torch",
        "from transformers import AutoModelForCausalLM, AutoTokenizer",
        "from peft import PeftModel",
        "m = AutoModelForCausalLM.from_pretrained('/content/v9-merged', dtype=torch.float16, device_map={'': 0})",
        "m = PeftModel.from_pretrained(m, '/content/v9-heretic').merge_and_unload()",
        "m.save_pretrained('/content/v9-heretic-merged', safe_serialization=True)",
        "AutoTokenizer.from_pretrained('/content/v9-heretic').save_pretrained('/content/v9-heretic-merged')",
        "print('merged adapter into full model')",
    ]
    Path("/content/_merge_adapter.py").write_text("\n".join(lines))
    subprocess.run([sys.executable, "/content/_merge_adapter.py"], check=True)
    SRC = Path("/content/v9-heretic-merged")
print("source for GGUF:", SRC)


In [ ]:
# 9. GGUF Q4_K_M (optional)
import subprocess, sys, os, shutil, urllib.request
from pathlib import Path

RUN_QUANTIZE = True
TAG = "b11734"
SRC = Path("/content/v9-heretic-merged") if Path("/content/v9-heretic-merged").exists() else Path("/content/v9-heretic")
OUT = Path("/content/v9-heretic-q4km.gguf")

if RUN_QUANTIZE:
    repo = Path("/content/llama.cpp")
    if not repo.exists():
        subprocess.run(["git","clone","--depth","1","--branch",TAG,
                        "https://github.com/ggml-org/llama.cpp", str(repo)], check=True)
    req = repo/"requirements"/"requirements-convert_hf_to_gguf.txt"
    subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(req)], check=True)
    tb = f"https://github.com/ggml-org/llama.cpp/releases/download/{TAG}/llama-{TAG}-bin-ubuntu-x64.tar.gz"
    urllib.request.urlretrieve(tb, "/tmp/llama.tgz")
    subprocess.run(["tar","xzf","/tmp/llama.tgz","-C","/content"], check=True)
    q = [str(p) for p in Path("/content").rglob("llama-quantize")
         if p.is_file() and os.access(p, os.X_OK)][0]

    subprocess.run([sys.executable, str(repo/"convert_hf_to_gguf.py"), str(SRC),
                    "--outfile","/content/v9-heretic-f16.gguf","--outtype","f16"], check=True)
    subprocess.run([q,"/content/v9-heretic-f16.gguf",str(OUT),"Q4_K_M"], check=True)
    os.remove("/content/v9-heretic-f16.gguf")
    print("GGUF: %.2f GB" % (OUT.stat().st_size/1e9))
    dst = Path("/content/drive/MyDrive/model-training/v9-heretic")/OUT.name
    shutil.copy2(OUT, dst)
    print("uploaded ->", dst)
